# Step 2 — 00. Checkpoint registration

목적은 공개 checkpoint의 출처, 로컬 SHA-256, 전처리, backbone,
학습 데이터 표기, 512D 출력과 Grad-CAM target layer를 하나의
불변 `ModelSpec`으로 등록하는 것입니다.

이 노트북은 모델을 학습하지 않습니다. 모델별 공식 구현·전처리·target
layer와 공통 crop의 RGB source order는 Step 2 설정에서 자동 선택합니다.
사용자는 실제 checkpoint 경로만 지정합니다. 기존 manifest는 덮어쓰지 않습니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("research와 configs가 있는 프로젝트 루트를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = True      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = True      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [2]:
from research.embeddings import (
    CheckpointProvenance,
    ModelSpec,
    PreprocessingSpec,
    write_model_spec,
)

CANDIDATE = CONFIG["models"]["candidates"][MODEL_NAME]

# 사용자가 지정할 값: 다운로드한 공식 checkpoint 파일
CHECKPOINT_PATH = None            # 예: PROJECT_ROOT / "models/arcface/model.pt"
CHECKPOINT_SOURCE_URL = CANDIDATE["checkpoint_source_page"]

# 모델별 구현 계약은 설정에서 자동 선택하며 이 셀에서 임의로 바꾸지 않는다.
SOURCE_COLOR_ORDER = CONFIG["aligned_crops"]["source_color_order"]
IMPLEMENTATION_REPOSITORY = CANDIDATE["implementation_repository"]
MODULE_FACTORY = CANDIDATE["loader_factory"]
TARGET_LAYER = CANDIDATE["target_layer"]
MODEL_COLOR_ORDER = CANDIDATE["preprocessing"]["model_color_order"]
CHANNEL_MEAN = tuple(CANDIDATE["preprocessing"]["mean"])
CHANNEL_STD = tuple(CANDIDATE["preprocessing"]["std"])

In [3]:
required = {
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "CHECKPOINT_SOURCE_URL": CHECKPOINT_SOURCE_URL,
}

if EXECUTE_STAGE:
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"checkpoint 등록값이 비어 있습니다: {missing}")
    checkpoint = CheckpointProvenance.from_file(
        CHECKPOINT_PATH,
        source_url=CHECKPOINT_SOURCE_URL,
    )
    preprocessing = PreprocessingSpec(
        input_height=CANDIDATE["preprocessing"]["input_size"][0],
        input_width=CANDIDATE["preprocessing"]["input_size"][1],
        source_color_order=SOURCE_COLOR_ORDER,
        model_color_order=MODEL_COLOR_ORDER,
        channel_mean=tuple(CHANNEL_MEAN),
        channel_std=tuple(CHANNEL_STD),
    )
    spec = ModelSpec(
        family=MODEL_NAME,
        architecture=CANDIDATE["backbone"],
        training_dataset=CANDIDATE["training_dataset"],
        implementation_repository=IMPLEMENTATION_REPOSITORY,
        checkpoint=checkpoint,
        preprocessing=preprocessing,
        target_layer=TARGET_LAYER,
        embedding_dim=CANDIDATE["embedding_dim"],
        module_factory=MODULE_FACTORY,
    )
    registration = spec.to_manifest()
    if WRITE_OUTPUTS:
        destination = (
            PROJECT_ROOT
            / "runs/step2/model_registry"
            / f"{spec.model_uid}.json"
        )
        write_model_spec(destination, spec)
        registration["written_to"] = str(destination)
else:
    registration = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
        "model_name": MODEL_NAME,
        "automatic_contract": {
            "architecture": CANDIDATE["backbone"],
            "loader_factory": MODULE_FACTORY,
            "target_layer": TARGET_LAYER,
            "model_color_order": MODEL_COLOR_ORDER,
            "mean": CHANNEL_MEAN,
            "std": CHANNEL_STD,
        },
    }
registration

RuntimeError: checkpoint 등록값이 비어 있습니다: ['CHECKPOINT_PATH']

다음 단계는 `01_preprocessing_and_model_smoke.ipynb`입니다. 새
checkpoint나 전처리 값으로 바꾸면 기존 manifest를 수정하지 말고 새
`model_uid`로 다시 등록합니다.